# 03 — Model Definition

Define SpecTNT, inspect architecture, test forward pass.

## Setup

In [ ]:
import sys, os
import torch
sys.path.insert(0, os.path.abspath(".."))
from utils.spectnt import SpecTNT, SpecTNTBlock, ResNetFrontEnd
from utils.losses import weighted_bce_loss, function_bce_loss, combined_loss, CTLLoss
device = torch.device("xpu" if torch.xpu.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


## Model architecture

In [ ]:
model = SpecTNT(dim=96, n_blocks=5).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,}")
print(f"Trainable:    {trainable:,}")
print(model)


## Forward pass test

In [ ]:
B, T_native = 4, 2205  # ~100 frames @ 21.5 fps
x = torch.randn(B, 1, 80, T_native, device=device)
b_logits, f_logits = model(x)
print(f"Input:           {list(x.shape)}")
print(f"Boundary logits: {list(b_logits.shape)}  (B, T', 1)")
print(f"Function logits: {list(f_logits.shape)}  (B, T', 7)")


## Loss function test

In [ ]:
b_target = torch.sigmoid(torch.randn_like(b_logits))
f_target = torch.sigmoid(torch.randn_like(f_logits))

loss = combined_loss(b_logits, b_target, f_logits, f_target)
print(f"Combined loss (no CTL): {loss.item():.4f}")

ctl = CTLLoss(blank=7)
B = b_logits.shape[0]
tokens = [torch.randint(0, 7, (5,)) for _ in range(B)]
il = torch.full((B,), f_logits.shape[1], dtype=torch.long)
tl = torch.tensor([5] * B, dtype=torch.long)
ctl_loss = ctl(f_logits, tokens, il, tl)
print(f"CTL loss: {ctl_loss.item():.4f}")


## Per-layer output shapes (smaller input to save memory)

In [ ]:
x_small = torch.randn(2, 1, 80, 516, device=device)  # ~24s native
x_feat = model.frontend(x_small)
print(f"After front-end: {list(x_feat.shape)}  (B, T', F', D)")
del x_small
for i, block in enumerate(model.blocks):
    x_feat = block(x_feat)
    print(f"After block {i+1}: {list(x_feat.shape)}")
x_pool = model.freq_pool(x_feat).squeeze(2)
print(f"After freq pool: {list(x_pool.shape)}  (B, T', D)")
x_out = model.proj(x_pool)
print(f"After proj: {list(x_out.shape)}  (B, T', 8)")
if hasattr(torch, "xpu") and torch.xpu.is_available():
    torch.xpu.empty_cache()
